In [39]:
import import_ipynb
from yahoo import getSP500_data
from cpi import get_cpi
from ffer import get_dff
from unemployment import get_unrate
from labor import get_civpart

In [40]:
cpi = get_cpi()
fedfunds = get_dff()
unrate = get_unrate()
civpart = get_civpart()
sp500 = getSP500_data()
sp500

,Date,First_Closing_Price,Final_Closing_Price,Monthly_Increase,Month,Year,Volatility
0,1927-12-31 00:00:00-05:00,17.660000,17.660000,0.000000,12,1927,NaN
1,1928-01-31 00:00:00-05:00,17.760000,17.570000,-0.190001,1,1928,0.121270
2,1928-02-29 00:00:00-05:00,17.530001,17.260000,-0.270000,2,1928,0.103970
3,1928-03-31 00:00:00-05:00,17.299999,19.280001,1.980001,3,1928,0.112672
4,1928-04-30 00:00:00-04:00,18.910000,19.750000,0.840000,4,1928,0.156696
...,...,...,...,...,...,...,...
1175,2025-11-30 00:00:00-05:00,6851.970215,6849.089844,-2.880371,11,2025,0.154245
1176,2025-12-31 00:00:00-05:00,6812.629883,6845.500000,32.870117,12,2025,0.088234
1177,2026-01-31 00:00:00-05:00,6858.470215,6939.029785,80.559570,1,2026,0.104286
1178,2026-02-28 00:00:00-05:00,6976.439941,6878.879883,-97.560059,2,2026,0.135263


In [41]:
keys = ["Month", "Year"]

def one_row_per_month_year(df):
    # Keep one record per Month/Year to avoid repeated rows after merge.
    if "Date" in df.columns:
        out = df.sort_values("Date").drop_duplicates(subset=keys, keep="last")
    elif "observation_date" in df.columns:
        out = df.sort_values("observation_date").drop_duplicates(subset=keys, keep="last")
    else:
        out = df.drop_duplicates(subset=keys, keep="last")

    # Prevent merge collisions from repeated date columns across sources.
    date_like_cols = [
        c for c in out.columns
        if c not in keys
        and (
            c.lower() in {"date", "observation_date", "observation_date_x", "observation_date_y"}
            or "observation_date" in c.lower()
            or c.lower().endswith("_date")
        )
    ]
    return out.drop(columns=date_like_cols, errors="ignore")

cpi_u = one_row_per_month_year(cpi)
fedfunds_u = one_row_per_month_year(fedfunds)
unrate_u = one_row_per_month_year(unrate)
sp500_u = one_row_per_month_year(sp500)
civpart_u = one_row_per_month_year(civpart)

merged_df = (
    sp500_u.merge(cpi_u, on=keys, how="inner")
           .merge(fedfunds_u, on=keys, how="inner")
           .merge(unrate_u, on=keys, how="inner")
           .merge(civpart_u, on=keys, how="inner")
)

# Final safety cleanup for any remaining date columns.
drop_cols = [
    c for c in merged_df.columns
    if c.lower() in {"date", "observation_date", "observation_date_x", "observation_date_y"}
    or "observation_date" in c.lower()
    or c.lower().endswith("_date")
]
merged_df = merged_df.drop(columns=drop_cols, errors="ignore")
merged_df = merged_df.drop(columns=["First_Closing_Price", "Final_Closing_Price"], errors="ignore")

merged_df.head()

,Monthly_Increase,Month,Year,Volatility,cpi_pct,fedfunds_rate,unemployment_rate,CIVPART
0,-3.870003,1,1968,0.068666,3.651861,4.75,3.7,59.2
1,-3.199997,2,1968,0.107240,3.673819,4.75,3.8,59.6
2,1.089996,3,1968,0.132372,4.142164,5.25,3.7,59.6
3,4.979996,4,1968,0.138902,4.155828,6.25,3.5,59.5
4,0.709999,5,1968,0.068541,4.088245,6.13,3.5,59.9


In [42]:

# Ensure chronological order first
merged_df = merged_df.sort_values(["Year", "Month"]).reset_index(drop=True)

# Make monthly increase one month ahead (t+1)
merged_df["Monthly_Increase_next_month"] = merged_df["Monthly_Increase"].shift(-1)

# Remove last row (it has no next month target)
merged_df = merged_df.dropna(subset=["Monthly_Increase_next_month"]).copy()

for col in ["cpi_pct", "fedfunds_rate", "unemployment_rate"]:
    merged_df[col + "_lag1"] = merged_df[col].shift(1)

merged_df.head()

,Monthly_Increase,Month,Year,Volatility,cpi_pct,fedfunds_rate,unemployment_rate,CIVPART,Monthly_Increase_next_month,cpi_pct_lag1,fedfunds_rate_lag1,unemployment_rate_lag1
0,-3.870003,1,1968,0.068666,3.651861,4.75,3.7,59.2,-3.199997,NaN,NaN,NaN
1,-3.199997,2,1968,0.107240,3.673819,4.75,3.8,59.6,1.089996,3.651861,4.75,3.7
2,1.089996,3,1968,0.132372,4.142164,5.25,3.7,59.6,4.979996,3.673819,4.75,3.8
3,4.979996,4,1968,0.138902,4.155828,6.25,3.5,59.5,0.709999,4.142164,5.25,3.7
4,0.709999,5,1968,0.068541,4.088245,6.13,3.5,59.9,-0.409996,4.155828,6.25,3.5


In [43]:
from pathlib import Path

download_path = Path.cwd().parent / "merged.csv"
merged_df.to_csv(download_path, index=False)